# Custom Eval Metrics

[Read the cookbook](https://docs.futureagi.com/docs/cookbook/quickstart/custom-eval-metrics). Register two rubrics, compare two support responses and score a code-review comment. Use a fresh Python 3.12 kernel. Hosted evaluations can consume credits.

## Install

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "agent-learning-kit==0.3.7"], check=True)

In [ ]:
from getpass import getpass
import os

for name in ('FI_API_KEY', 'FI_SECRET_KEY'):
    if not os.environ.get(name):
        os.environ[name] = getpass(f"{name}: ")
    if not os.environ[name].strip():
        raise ValueError(f"{name} is required")

## Connect to your workspace

In [ ]:
import json
from pathlib import Path
from uuid import uuid4
from fi.evals import Evaluator, EvalTemplateManager

manager = EvalTemplateManager()
evaluator = Evaluator()
suffix = uuid4().hex[:8]

## Configure a Pass/Fail support-quality eval

In [ ]:
support = manager.create_template(
    name=f"support_quality_{suffix}",
    model="turing_small",
    output_type="pass_fail",
    instructions="""You are evaluating a customer support response.

The customer asked: {{user_query}}
The agent responded: {{agent_response}}

Mark PASS only if all of these are true:
- It acknowledges the customer's specific issue
- It gives a concrete next step or resolution
- It maintains a professional and empathetic tone

Mark FAIL if any required condition is missing, or if the response is dismissive, vague, or off-topic.

Return a clear PASS/FAIL decision with a short reason.""",
)
print(support.name)

## Run the support-quality eval against a weak response

In [ ]:
result = evaluator.evaluate(
    eval_templates=support.name,
    model_name="turing_small",
    inputs={
        "user_query": "My order arrived damaged. What do I do?",
        "agent_response": "Please contact our returns department.",
    },
)

weak = result.eval_results[0]
print(weak.output)
print(weak.reason)

## Confirm the rubric passes a fixed response

In [ ]:
result = evaluator.evaluate(
    eval_templates=support.name,
    model_name="turing_small",
    inputs={
        "user_query": "My order arrived damaged. What do I do?",
        "agent_response": "I'm sorry to hear that. I've filed a replacement request and you'll receive a shipping confirmation within 24 hours.",
    },
)

fixed = result.eval_results[0]
print(fixed.output)
print(fixed.reason)

## Configure a Percentage code-review eval

In [ ]:
review = manager.create_template(
    name=f"code_review_quality_{suffix}",
    model="turing_small",
    output_type="percentage",
    instructions="""You are evaluating a code review comment.

The code change: {{code_diff}}
The review comment: {{review_comment}}

Score using these weights:
- 40 points: Does it clearly explain what's wrong?
- 30 points: Does it suggest a concrete fix or improvement?
- 30 points: Is it constructive and respectful?

Return a normalized score from 0.0 to 1.0 (for example, 0.91 for 91/100).""",
)
print(review.name)

## Run the code-review eval from Python

In [ ]:
result = evaluator.evaluate(
    eval_templates=review.name,
    model_name="turing_small",
    inputs={
        "code_diff": "- return user.name\n+ return user.name.strip()",
        "review_comment": "Good catch: whitespace in names can cause login failures. Consider adding a test case for this.",
    },
)

eval_result = result.eval_results[0]
print(f"Score: {eval_result.output}")
print(f"Reason: {eval_result.reason}")

# Keep the IDs if you want to revisit these templates and their results.
Path("rubric-runs.json").write_text(json.dumps({
    "support": {"id": support.id, "name": support.name, "runs": [weak.eval_id, fixed.eval_id]},
    "review": {"id": review.id, "name": review.name, "runs": [eval_result.eval_id]},
}, indent=2))

Search **Evals** in your workspace for the printed template names. `rubric-runs.json` keeps both template IDs and all three run IDs.